# State-aware two-pass feature selection for IoT gateway flow monitoring

This notebook reproduces every table and figure of the study. It selects the **per-flow state a gateway
flow collector has to keep**, rather than a number of classifier inputs, and measures what that state costs.

**Before running**

1. Download the RT-IoT2022 CSV from the UCI Machine Learning Repository (https://doi.org/10.24432/C5P338).
2. In Colab, place it at `MyDrive/Datasets/RT_IOT2022.csv`. Elsewhere, set the environment variables
   `GW_DATA` (CSV path), `GW_WORK` (scratch folder) and `GW_OUT` (results folder).
3. Run all cells from top to bottom. No GPU is needed. The full evaluation takes roughly one hour on two
   CPU cores; set `QUICK_TEST = True` in the first cell for a five-minute installation check.

**Where files go.** Models, compiled collectors and intermediate files stay on the local disk. Only small
result files (CSV tables, figures, two small files per fold) are copied to the results folder, so an
interrupted run resumes from the completed folds.

**Requirements.** Python with scikit-learn, pandas, SciPy, matplotlib, SHAP, and `gcc` for the collector
benchmark. The versions actually used are written to `Tables/run_metadata.json`.

In [1]:
# ---- Paths and parameters -------------------------------------------------------
import os, shutil, subprocess, sys
from pathlib import Path

QUICK_TEST = False          # True: one fold and two random orderings, to check the installation
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/Datasets/RT_IOT2022.csv")           # read only
    WORK = Path("/content/gateway_state")                                    # local disk
    OUT = Path("/content/drive/MyDrive/Outputs/FGCS_Gateway_State_Selection")  # small results only
except ImportError:
    DATA = Path(os.environ.get("GW_DATA", "RT_IOT2022.csv"))
    WORK = Path(os.environ.get("GW_WORK", "gateway_state_work"))
    OUT = Path(os.environ.get("GW_OUT", "gateway_state_results"))
import importlib.util, warnings
if importlib.util.find_spec("shap") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
warnings.filterwarnings("ignore", message="IProgress not found")
TABLES, FIGURES = WORK / "Tables", WORK / "Figures"
for d in (WORK, OUT, TABLES, FIGURES, WORK / "folds"):
    d.mkdir(parents=True, exist_ok=True)
for saved in (OUT / "folds").glob("s*_f*.*"):          # resume from the small backups on Drive
    if not (WORK / "folds" / saved.name).exists():
        shutil.copy(saved, WORK / "folds" / saved.name)

SEEDS, N_FOLDS = (42, 43, 44), 5
N_TREES = 100                      # evaluation forests
SHAP_ROWS = 504                    # class-balanced rows for pass-1 attributions
MI_ROWS_PER_CLASS = 2000
TOP_K = 20                         # size of the fixed top-k design used as a reference point
BUDGETS = [0, 2, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 384]   # bytes of per-flow state
DELTAS, PRIMARY_DELTA = (0.02, 0.01, 0.005), 0.01    # stopping tolerances on out-of-bag macro-F1
GREEDY_CAP, GREEDY_TREES = 4000, 50                  # rows per class and trees used to score a move
SHORTLIST, WORST_CLASSES, MIN_GAIN, MAX_MEASURED = 4, 2, 0.001, 20
N_RANDOM = 2 if QUICK_TEST else 6                    # random orderings per fold
MAX_FOLDS = 1 if QUICK_TEST else len(SEEDS) * N_FOLDS
N_BOOT, BOOT_BUDGETS = (50 if QUICK_TEST else 1000), [8, 16, 32, 64]
BENCH_FLOWS, BENCH_PACKETS, BENCH_REPS, BENCH_ROUNDS = 65536, 2_000_000, 7, (1 if QUICK_TEST else 5)
print(f"data: {DATA}\nwork: {WORK}\nresults: {OUT}\nquick test: {QUICK_TEST}")

data: RT_IOT2022.csv
work: gateway_state_work
results: gateway_state_results
quick test: False


## 1. Collector-state model

Every flow feature of the dataset is produced by the Zeek FlowMeter script. The table below lists the
per-flow state variables (*primitives*) that a constant-memory streaming collector needs in order to
reproduce those features, and maps each feature to the primitives it requires. Features share primitives,
so the cost of a feature set is the size of the **union** of its primitives.

In [2]:
# ---- Collector-state model -------------------------------------------------
# Each primitive is one per-flow state variable of a constant-memory streaming
# collector that reproduces the Zeek FlowMeter feature definitions
# (zeek-flowmeter, scripts/flowmeter.zeek). Width in bytes is the C type used
# by the generated collector in the next section.
PRIM = {  # name: (C type, bytes, description)
    "n_f": ("uint32_t", 4, "forward packet count"),
    "n_b": ("uint32_t", 4, "backward packet count"),
    "hs_f": ("uint32_t", 4, "forward header bytes, sum"),
    "hn_f": ("uint8_t", 1, "forward header bytes, min"),
    "hx_f": ("uint8_t", 1, "forward header bytes, max"),
    "hs_b": ("uint32_t", 4, "backward header bytes, sum"),
    "hn_b": ("uint8_t", 1, "backward header bytes, min"),
    "hx_b": ("uint8_t", 1, "backward header bytes, max"),
    "fin": ("uint32_t", 4, "FIN count"), "syn": ("uint32_t", 4, "SYN count"),
    "rst": ("uint32_t", 4, "RST count"), "ack": ("uint32_t", 4, "ACK count"),
    "psh_f": ("uint32_t", 4, "forward PSH count"), "psh_b": ("uint32_t", 4, "backward PSH count"),
    "urg_f": ("uint32_t", 4, "forward URG count"), "urg_b": ("uint32_t", 4, "backward URG count"),
    "cwr": ("uint32_t", 4, "CWR count"), "ece": ("uint32_t", 4, "ECE count"),
    "d_f": ("uint32_t", 4, "forward data-packet count"),
    "d_b": ("uint32_t", 4, "backward data-packet count"),
    "ps_f": ("uint64_t", 8, "forward payload bytes, sum"),
    "pq_f": ("uint64_t", 8, "forward payload bytes, sum of squares"),
    "pn_f": ("uint16_t", 2, "forward payload bytes, min"),
    "px_f": ("uint16_t", 2, "forward payload bytes, max"),
    "ps_b": ("uint64_t", 8, "backward payload bytes, sum"),
    "pq_b": ("uint64_t", 8, "backward payload bytes, sum of squares"),
    "pn_b": ("uint16_t", 2, "backward payload bytes, min"),
    "px_b": ("uint16_t", 2, "backward payload bytes, max"),
    "t_f": ("uint64_t", 8, "time of last forward packet"),
    "t_b": ("uint64_t", 8, "time of last backward packet"),
    "is_f": ("uint64_t", 8, "forward inter-arrival, sum"),
    "iq_f": ("double", 8, "forward inter-arrival, sum of squares"),
    "in_f": ("uint64_t", 8, "forward inter-arrival, min"),
    "ix_f": ("uint64_t", 8, "forward inter-arrival, max"),
    "is_b": ("uint64_t", 8, "backward inter-arrival, sum"),
    "iq_b": ("double", 8, "backward inter-arrival, sum of squares"),
    "in_b": ("uint64_t", 8, "backward inter-arrival, min"),
    "ix_b": ("uint64_t", 8, "backward inter-arrival, max"),
    "iq": ("double", 8, "flow inter-arrival, sum of squares"),
    "in_": ("uint64_t", 8, "flow inter-arrival, min"),
    "ix": ("uint64_t", 8, "flow inter-arrival, max"),
    "sub": ("uint32_t", 4, "subflow count"),
    "a_new": ("uint8_t", 1, "active phase: new/open flags"),
    "a_cur": ("uint64_t", 8, "active phase: running length"),
    "a_n": ("uint32_t", 4, "active phases, count"),
    "a_s": ("uint64_t", 8, "active phases, sum"),
    "a_q": ("double", 8, "active phases, sum of squares"),
    "a_mn": ("uint64_t", 8, "active phases, min"),
    "a_mx": ("uint64_t", 8, "active phases, max"),
    "l_n": ("uint32_t", 4, "idle periods, count"),
    "l_s": ("uint64_t", 8, "idle periods, sum"),
    "l_q": ("double", 8, "idle periods, sum of squares"),
    "l_mn": ("uint64_t", 8, "idle periods, min"),
    "l_mx": ("uint64_t", 8, "idle periods, max"),
    "pwf": ("uint8_t", 1, "bulk: previous data packet was forward"),
    "btp_f": ("uint32_t", 4, "forward bulk candidate, packets"),
    "btb_f": ("uint32_t", 4, "forward bulk candidate, bytes"),
    "btt_f": ("uint64_t", 8, "forward bulk candidate, duration"),
    "bc_f": ("uint32_t", 4, "forward bulks, count"),
    "bp_f": ("uint32_t", 4, "forward bulks, packets"),
    "bb_f": ("uint64_t", 8, "forward bulks, bytes"),
    "bt_f": ("uint64_t", 8, "forward bulks, duration"),
    "btp_b": ("uint32_t", 4, "backward bulk candidate, packets"),
    "btb_b": ("uint32_t", 4, "backward bulk candidate, bytes"),
    "btt_b": ("uint64_t", 8, "backward bulk candidate, duration"),
    "bc_b": ("uint32_t", 4, "backward bulks, count"),
    "bp_b": ("uint32_t", 4, "backward bulks, packets"),
    "bb_b": ("uint64_t", 8, "backward bulks, bytes"),
    "bt_b": ("uint64_t", 8, "backward bulks, duration"),
    "wi_f": ("uint16_t", 2, "forward initial TCP window"),
    "wl_f": ("uint16_t", 2, "forward last TCP window"),
    "wi_b": ("uint16_t", 2, "backward initial TCP window"),
}
PRIM_BYTES = {k: v[1] for k, v in PRIM.items()}


def _stats(prefix, s, q, mn, mx, n):
    """tot/avg/std/min/max of an accumulator family -> primitive sets."""
    return {f"{prefix}.min": mn, f"{prefix}.max": mx, f"{prefix}.tot": s,
            f"{prefix}.avg": s + n, f"{prefix}.std": s + q + n}


# feature -> primitives it needs. Features with an empty list are computable
# from the flow key and the first/last packet times that any flow table keeps.
NEEDS = {
    "proto": [], "flow_duration": [],
    "fwd_pkts_tot": ["n_f"], "bwd_pkts_tot": ["n_b"],
    "fwd_data_pkts_tot": ["d_f"], "bwd_data_pkts_tot": ["d_b"],
    "fwd_pkts_per_sec": ["n_f"], "bwd_pkts_per_sec": ["n_b"],
    "flow_pkts_per_sec": ["n_f", "n_b"], "down_up_ratio": ["n_f", "n_b"],
    "fwd_header_size_tot": ["hs_f"], "fwd_header_size_min": ["hn_f"],
    "fwd_header_size_max": ["hx_f"], "bwd_header_size_tot": ["hs_b"],
    "bwd_header_size_min": ["hn_b"], "bwd_header_size_max": ["hx_b"],
    "flow_FIN_flag_count": ["fin"], "flow_SYN_flag_count": ["syn"],
    "flow_RST_flag_count": ["rst"], "fwd_PSH_flag_count": ["psh_f"],
    "bwd_PSH_flag_count": ["psh_b"], "flow_ACK_flag_count": ["ack"],
    "fwd_URG_flag_count": ["urg_f"], "bwd_URG_flag_count": ["urg_b"],
    "flow_CWR_flag_count": ["cwr"], "flow_ECE_flag_count": ["ece"],
    **_stats("fwd_pkts_payload", ["ps_f"], ["pq_f"], ["pn_f"], ["px_f"], ["n_f"]),
    **_stats("bwd_pkts_payload", ["ps_b"], ["pq_b"], ["pn_b"], ["px_b"], ["n_b"]),
    **_stats("flow_pkts_payload", ["ps_f", "ps_b"], ["pq_f", "pq_b"],
             ["pn_f", "pn_b"], ["px_f", "px_b"], ["n_f", "n_b"]),
    **_stats("fwd_iat", ["t_f", "is_f"], ["iq_f"], ["t_f", "in_f"], ["t_f", "ix_f"], ["n_f"]),
    **_stats("bwd_iat", ["t_b", "is_b"], ["iq_b"], ["t_b", "in_b"], ["t_b", "ix_b"], ["n_b"]),
    **_stats("flow_iat", [], ["iq"], ["in_"], ["ix"], ["n_f", "n_b"]),
    "payload_bytes_per_second": ["ps_f", "ps_b"],
    "fwd_subflow_pkts": ["n_f", "sub"], "bwd_subflow_pkts": ["n_b", "sub"],
    "fwd_subflow_bytes": ["ps_f", "sub"], "bwd_subflow_bytes": ["ps_b", "sub"],
    "fwd_bulk_bytes": ["pwf", "t_f", "btp_f", "btb_f", "bc_f", "bb_f"],
    "bwd_bulk_bytes": ["pwf", "t_b", "btp_b", "btb_b", "bc_b", "bb_b"],
    "fwd_bulk_packets": ["pwf", "t_f", "btp_f", "bc_f", "bp_f"],
    "bwd_bulk_packets": ["pwf", "t_b", "btp_b", "bc_b", "bp_b"],
    "fwd_bulk_rate": ["pwf", "t_f", "btp_f", "btb_f", "btt_f", "bb_f", "bt_f"],
    "bwd_bulk_rate": ["pwf", "t_b", "btp_b", "btb_b", "btt_b", "bb_b", "bt_b"],
    "active.min": ["a_new", "a_cur", "a_mn"], "active.max": ["a_new", "a_cur", "a_mx"],
    "active.tot": ["a_s"], "active.avg": ["a_s", "a_new", "a_n"],
    "active.std": ["a_s", "a_new", "a_cur", "a_n", "a_q"],
    "idle.min": ["l_mn"], "idle.max": ["l_mx"], "idle.tot": ["l_s"],
    "idle.avg": ["l_s", "l_n"], "idle.std": ["l_s", "l_q", "l_n"],
    "fwd_init_window_size": ["n_f", "wi_f"], "fwd_last_window_size": ["wl_f"],
    "bwd_init_window_size": ["n_b", "wi_b"],
}
NEEDS = {f: frozenset(p) for f, p in NEEDS.items()}
# Fields that identify endpoints or come from payload inspection. They need no
# flow-statistics state and are analysed separately as potential shortcuts.
IDENTITY = ["id.orig_p", "id.resp_p", "service"]
FULL_STATE_BYTES = sum(PRIM_BYTES.values())


def state_bytes(prims):
    return sum(PRIM_BYTES[p] for p in prims)


def prims_of(features):
    out = set()
    for f in features:
        out |= NEEDS.get(f, frozenset())
    return frozenset(out)


def unlocked(prims, universe):
    """All features of the universe computable from a primitive set."""
    return [f for f in universe if NEEDS[f] <= prims]

## 2. A collector that can be sliced

The C collector is generated from statements tagged with the primitives they touch. Asking for a subset of
primitives emits only the matching statements and struct fields, so each configuration is a slice of one
program. The benchmark replays synthetic interleaved packets through the per-packet update routine and
reports nanoseconds per packet and the size of the per-flow record.

In [3]:
# ---- Generated streaming collector -----------------------------------------
# The C source below is assembled from statements tagged with the primitives
# they touch. A statement is emitted only when all of its primitives belong to
# the requested configuration, so each configuration is a slice of the same
# program. Semantics follow scripts/flowmeter.zeek statement by statement,
# with running accumulators in place of the per-flow vectors.
import os, struct, subprocess, tempfile

SUBFLOW_US, ACTIVE_US, BULK_US, BULK_MIN = 1_000_000, 5_000_000, 1_000_000, 5
FEATURES_ALL = list(NEEDS)  # output order of the collector


def _g(have, need, code):
    return code if set(need) <= have else ""


def collector_source(prims):
    have = set(prims)
    order = sorted(have, key=lambda p: (-PRIM_BYTES[p], p))  # widest first: no padding holes
    fields = "".join(f"  {PRIM[p][0]} {p};\n" for p in order)
    u = []  # per-packet update statements
    A = lambda need, code: u.append(_g(have, need, code))
    A(["n_f"], "if (fwd) s->n_f++;")
    A(["n_b"], "if (!fwd) s->n_b++;")
    for d, c in (("f", "fwd"), ("b", "!fwd")):
        A([f"hs_{d}"], f"if ({c}) s->hs_{d} += p->hdr;")
        A([f"hn_{d}"], f"if ({c} && (s->hn_{d} > p->hdr || s->hn_{d} == 0)) s->hn_{d} = p->hdr;")
        A([f"hx_{d}"], f"if ({c} && s->hx_{d} < p->hdr) s->hx_{d} = p->hdr;")
        A([f"ps_{d}"], f"if ({c}) s->ps_{d} += p->pay;")
        A([f"pq_{d}"], f"if ({c}) s->pq_{d} += (uint64_t)p->pay * p->pay;")
        A([f"pn_{d}"], f"if ({c} && p->pay < s->pn_{d}) s->pn_{d} = p->pay;")
        A([f"px_{d}"], f"if ({c} && p->pay > s->px_{d}) s->px_{d} = p->pay;")
        A([f"d_{d}"], f"if ({c} && p->pay > 0) s->d_{d}++;")
    for name, bit in (("fin", 1), ("syn", 2), ("rst", 4), ("ack", 16), ("ece", 64), ("cwr", 128)):
        A([name], f"if (p->tcp && (p->flags & {bit})) s->{name}++;")
    for name, bit in (("psh", 8), ("urg", 32)):
        A([f"{name}_f"], f"if (p->tcp && (p->flags & {bit}) && fwd) s->{name}_f++;")
        A([f"{name}_b"], f"if (p->tcp && (p->flags & {bit}) && !fwd) s->{name}_b++;")
    # inter-arrival times; the flow value is the gap to the previous packet
    A(["t_f"], "const int has_f = s->t_f != 0; const uint64_t iat_f = now - s->t_f;")
    A(["t_b"], "const int has_b = s->t_b != 0; const uint64_t iat_b = now - s->t_b;")
    flow = []
    F = lambda need, code: flow.append(_g(have, need, code))
    F(["sub"], f"if (iat > {SUBFLOW_US}ULL) s->sub++;")
    idle = "".join([
        _g(have, ["l_n"], "s->l_n++;"), _g(have, ["l_s"], "s->l_s += iat;"),
        _g(have, ["l_q"], "s->l_q += (double)iat * (double)iat;"),
        _g(have, ["l_mn"], "if (iat < s->l_mn) s->l_mn = iat;"),
        _g(have, ["l_mx"], "if (iat > s->l_mx) s->l_mx = iat;"),
        _g(have, ["a_new"], "s->a_new |= 1;")])
    close = "".join([
        _g(have, ["a_cur", "a_q"], "s->a_q += (double)s->a_cur * (double)s->a_cur;"),
        _g(have, ["a_cur", "a_mn"], "if (s->a_cur < s->a_mn) s->a_mn = s->a_cur;"),
        _g(have, ["a_cur", "a_mx"], "if (s->a_cur > s->a_mx) s->a_mx = s->a_cur;")])
    start = "".join([
        _g(have, ["a_new"], f"if (s->a_new & 2) {{ {close} }}"),
        _g(have, ["a_new", "a_n"], "s->a_n++;"),
        _g(have, ["a_new", "a_cur"], "s->a_cur = iat;"),
        _g(have, ["a_new"], "s->a_new = 2;")])
    cont = _g(have, ["a_new", "a_cur"], "s->a_cur += iat;")
    act_sum = _g(have, ["a_s"], "s->a_s += iat;")
    if "a_new" in have:
        flow.append(f"if (iat > {ACTIVE_US}ULL) {{ {idle} }} else {{ {act_sum} "
                    f"if (s->a_new & 1) {{ {start} }} else {{ {cont} }} }}")
    else:
        flow.append(f"if (iat > {ACTIVE_US}ULL) {{ {idle} }} else {{ {act_sum} }}")
    F(["iq"], "s->iq += (double)iat * (double)iat;")
    F(["in_"], "if (iat < s->in_) s->in_ = iat;")
    F(["ix"], "if (iat > s->ix) s->ix = iat;")
    u.append("if (s->seen) { const uint64_t iat = now - s->t_last; (void)iat; "
             + " ".join(x for x in flow if x) + " }")
    for d, c in (("f", "fwd"), ("b", "!fwd")):
        inner = "".join([
            _g(have, [f"is_{d}"], f"s->is_{d} += iat_{d};"),
            _g(have, [f"iq_{d}"], f"s->iq_{d} += (double)iat_{d} * (double)iat_{d};"),
            _g(have, [f"in_{d}"], f"if (iat_{d} < s->in_{d}) s->in_{d} = iat_{d};"),
            _g(have, [f"ix_{d}"], f"if (iat_{d} > s->ix_{d}) s->ix_{d} = iat_{d};")])
        if inner and f"t_{d}" in have:
            u.append(f"if ({c} && has_{d}) {{ {inner} }}")
    # bulk transmissions
    def flush(d):
        body = "".join([
            _g(have, [f"bc_{d}"], f"s->bc_{d}++;"),
            _g(have, [f"bp_{d}"], f"s->bp_{d} += s->btp_{d};"),
            _g(have, [f"bb_{d}", f"btb_{d}"], f"s->bb_{d} += s->btb_{d};"),
            _g(have, [f"bt_{d}", f"btt_{d}"], f"s->bt_{d} += s->btt_{d};")])
        return _g(have, [f"btp_{d}"], f"if (s->btp_{d} >= {BULK_MIN}) {{ {body} }}")
    def reset(d):
        return "".join([
            _g(have, [f"btp_{d}"], f"s->btp_{d} = 1;"),
            _g(have, [f"btb_{d}"], f"s->btb_{d} = p->pay;"),
            _g(have, [f"btt_{d}"], f"s->btt_{d} = 0;")])
    def grow(d):
        return "".join([
            _g(have, [f"btp_{d}"], f"s->btp_{d}++;"),
            _g(have, [f"btb_{d}"], f"s->btb_{d} += p->pay;"),
            _g(have, [f"btt_{d}"], f"s->btt_{d} += iat_{d};")])
    if "pwf" in have:
        def side(d, o, same):
            if f"btp_{d}" in have and f"t_{d}" in have:
                own = (f"if ({same}) {{ if (has_{d} && iat_{d} < {BULK_US}ULL) {{ {grow(d)} }} "
                       f"else {{ {flush(d)} {reset(d)} }} }} else {{ {flush(o)} {reset(d)} }}")
            else:
                own = f"if (!({same})) {{ {flush(o)} }}"
            return own
        u.append("if (p->pay > 0) { if (fwd) { " + side("f", "b", "s->pwf")
                 + " s->pwf = 1; } else { " + side("b", "f", "!s->pwf") + " s->pwf = 0; } }")
    A(["n_f", "wi_f"], "if (p->tcp && fwd && s->n_f == 1) s->wi_f = p->win;")
    A(["wl_f"], "if (p->tcp && fwd) s->wl_f = p->win;")
    A(["n_b", "wi_b"], "if (p->tcp && !fwd && s->n_b == 1) s->wi_b = p->win;")
    A(["t_f"], "if (fwd) s->t_f = now;")
    A(["t_b"], "if (!fwd) s->t_b = now;")
    init = "".join(_g(have, [p], f"s->{p} = {v};") for p, v in (
        ("pn_f", "0xFFFF"), ("pn_b", "0xFFFF"), ("in_f", "UINT64_MAX"), ("in_b", "UINT64_MAX"),
        ("in_", "UINT64_MAX"), ("a_mn", "UINT64_MAX"), ("l_mn", "UINT64_MAX"),
        ("sub", "1"), ("a_new", "1")))
    # end-of-flow feature computation
    e = []
    def stat(prefix, n, s_, q, mn, mx, feats, sentinel="UINT64_MAX"):
        for key, expr in (
            ("min", f"(({mn}) == {sentinel} ? 0.0 : (double)({mn}))"),  # sentinel: nothing observed
            ("max", f"(double)({mx})"),
            ("tot", f"(double)({s_})"),
            ("avg", f"(({n}) > 0 ? (double)({s_}) / (double)({n}) : 0.0)"),
            ("std", f"sd((double)({n}), (double)({s_}), (double)({q}))")):
            name = f"{prefix}.{key}"
            if name in feats:
                e.append(f"o[{FEATURES_ALL.index(name)}] = {expr};")
    feats = set(unlocked(frozenset(have), FEATURES_ALL))
    simple = {
        "flow_duration": "dur", "fwd_pkts_tot": "s->n_f", "bwd_pkts_tot": "s->n_b",
        "fwd_data_pkts_tot": "s->d_f", "bwd_data_pkts_tot": "s->d_b",
        "fwd_pkts_per_sec": "(dur > 0 ? s->n_f / dur : 0.0)",
        "bwd_pkts_per_sec": "(dur > 0 ? s->n_b / dur : 0.0)",
        "flow_pkts_per_sec": "(dur > 0 ? ((double)s->n_f + s->n_b) / dur : 0.0)",
        "down_up_ratio": "(s->n_f > 0 ? (double)s->n_b / s->n_f : 0.0)",
        "fwd_header_size_tot": "s->hs_f", "fwd_header_size_min": "s->hn_f",
        "fwd_header_size_max": "s->hx_f", "bwd_header_size_tot": "s->hs_b",
        "bwd_header_size_min": "s->hn_b", "bwd_header_size_max": "s->hx_b",
        "flow_FIN_flag_count": "s->fin", "flow_SYN_flag_count": "s->syn",
        "flow_RST_flag_count": "s->rst", "fwd_PSH_flag_count": "s->psh_f",
        "bwd_PSH_flag_count": "s->psh_b", "flow_ACK_flag_count": "s->ack",
        "fwd_URG_flag_count": "s->urg_f", "bwd_URG_flag_count": "s->urg_b",
        "flow_CWR_flag_count": "s->cwr", "flow_ECE_flag_count": "s->ece",
        "payload_bytes_per_second": "(dur > 0 ? ((double)s->ps_f + (double)s->ps_b) / dur : 0.0)",
        "fwd_subflow_pkts": "(double)s->n_f / s->sub", "bwd_subflow_pkts": "(double)s->n_b / s->sub",
        "fwd_subflow_bytes": "(double)s->ps_f / s->sub", "bwd_subflow_bytes": "(double)s->ps_b / s->sub",
        "fwd_init_window_size": "s->wi_f", "fwd_last_window_size": "s->wl_f",
        "bwd_init_window_size": "s->wi_b",
    }
    for d in "fb":
        w = "fwd" if d == "f" else "bwd"
        simple[f"{w}_bulk_bytes"] = f"(s->bc_{d} > 0 ? (double)s->bb_{d} / s->bc_{d} : 0.0)"
        simple[f"{w}_bulk_packets"] = f"(s->bc_{d} > 0 ? (double)s->bp_{d} / s->bc_{d} : 0.0)"
        simple[f"{w}_bulk_rate"] = f"(s->bt_{d} > 0 ? (double)s->bb_{d} / ((double)s->bt_{d} / 1e6) : 0.0)"
    for name, expr in simple.items():
        if name in feats:
            e.append(f"o[{FEATURES_ALL.index(name)}] = (double)({expr});")
    for d, w in (("f", "fwd"), ("b", "bwd")):
        stat(f"{w}_pkts_payload", f"s->n_{d}", f"s->ps_{d}", f"s->pq_{d}", f"s->pn_{d}", f"s->px_{d}", feats, "0xFFFF")
        stat(f"{w}_iat", f"(s->n_{d} > 0 ? s->n_{d} - 1 : 0)", f"s->is_{d}", f"s->iq_{d}",
             f"s->in_{d}", f"s->ix_{d}", feats)
    stat("flow_pkts_payload", "(uint64_t)s->n_f + s->n_b", "s->ps_f + s->ps_b", "s->pq_f + s->pq_b",
         "(s->pn_f < s->pn_b ? s->pn_f : s->pn_b)", "(s->px_f > s->px_b ? s->px_f : s->px_b)", feats, "0xFFFF")
    stat("flow_iat", "((uint64_t)s->n_f + s->n_b > 0 ? (uint64_t)s->n_f + s->n_b - 1 : 0)",
         "(s->t_last - s->t_first)", "s->iq", "s->in_", "s->ix", feats)
    stat("active", "s->a_n", "s->a_s", "s->a_q", "s->a_mn", "s->a_mx", feats)
    stat("idle", "s->l_n", "s->l_s", "s->l_q", "s->l_mn", "s->l_mx", feats)
    fin = "".join([flush("f"), flush("b"), _g(have, ["a_new"], f"if (s->a_new & 2) {{ {close} }}")])
    return f"""
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>
typedef struct {{ uint64_t ts; uint32_t flow; uint16_t pay; uint16_t win;
                 uint8_t fwd, tcp, flags, hdr; }} pkt_t;
typedef struct {{
  uint64_t t_first, t_last;
{fields}  uint8_t seen;
}} flow_t;
#define NFEAT {len(FEATURES_ALL)}
static inline double sd(double n, double s, double q) {{
  if (n < 2) return 0.0;
  double v = (q - s * s / n) / (n - 1);
  return v > 0 ? sqrt(v) : 0.0;
}}
static inline void update(flow_t *s, const pkt_t *p) {{
  const uint64_t now = p->ts; const int fwd = p->fwd;
  if (!s->seen) {{ s->t_first = now; {init} }}
  {chr(10).join("  " + x for x in u if x)}
  s->t_last = now; s->seen = 1;
}}
static void finish(flow_t *s, double *o) {{
  {fin}
  const double dur = (double)(s->t_last - s->t_first) / 1e6; (void)dur;
  {chr(10).join("  " + x for x in e)}
}}
static uint64_t rs = 88172645463325252ULL;
static inline uint64_t rnd(void) {{ rs ^= rs << 13; rs ^= rs >> 7; rs ^= rs << 17; return rs; }}
int main(int argc, char **argv) {{
  if (argc > 1 && !strcmp(argv[1], "size")) {{ printf("%zu\\n", sizeof(flow_t)); return 0; }}
  if (argc > 3 && !strcmp(argv[1], "feat")) {{       /* feat trace.bin out.bin nflows */
    long nf = atol(argv[4]); FILE *f = fopen(argv[2], "rb"); pkt_t p;
    flow_t *t = calloc(nf, sizeof(flow_t)); double *o = calloc(NFEAT, sizeof(double));
    while (fread(&p, sizeof p, 1, f) == 1) update(&t[p.flow], &p);
    fclose(f); f = fopen(argv[3], "wb");
    for (long i = 0; i < nf; i++) {{ memset(o, 0, NFEAT * sizeof(double)); if (t[i].seen) finish(&t[i], o);
      fwrite(o, sizeof(double), NFEAT, f); }}
    fclose(f); return 0;
  }}
  /* bench nflows npkts reps: synthetic interleaved packets, median ns per packet */
  long nf = atol(argv[2]), np = atol(argv[3]); int reps = atoi(argv[4]);
  pkt_t *pk = malloc(np * sizeof(pkt_t)); flow_t *t = malloc(nf * sizeof(flow_t));
  uint64_t now = 1000;
  for (long i = 0; i < np; i++) {{
    uint64_t r = rnd(); pkt_t *p = &pk[i];
    now += 1 + (r & 63); p->ts = now; p->flow = (uint32_t)((r >> 8) % nf);
    p->fwd = (r >> 40) & 1; p->tcp = ((r >> 41) & 7) != 0;
    p->flags = (uint8_t)(r >> 44); p->hdr = p->tcp ? 20 + 4 * ((r >> 52) & 3) : 8;
    p->pay = ((r >> 54) & 3) ? (uint16_t)((r >> 20) % 1400) : 0; p->win = (uint16_t)(r >> 24);
  }}
  double best[64]; double sink = 0;
  for (int k = 0; k < reps; k++) {{
    memset(t, 0, nf * sizeof(flow_t)); struct timespec a, b;
    clock_gettime(CLOCK_MONOTONIC, &a);
    for (long i = 0; i < np; i++) update(&t[pk[i].flow], &pk[i]);
    clock_gettime(CLOCK_MONOTONIC, &b);
    best[k] = ((b.tv_sec - a.tv_sec) * 1e9 + (b.tv_nsec - a.tv_nsec)) / np;
    sink += (double)t[k % nf].t_last;
  }}
  for (int i = 0; i < reps; i++) for (int j = i + 1; j < reps; j++)
    if (best[j] < best[i]) {{ double x = best[i]; best[i] = best[j]; best[j] = x; }}
  printf("%.3f %.3f %zu %.0f\\n", best[reps / 2], best[0], sizeof(flow_t), sink);
  return 0;
}}
"""


_BIN_CACHE = {}


def build_collector(prims, workdir):
    key = frozenset(prims)
    if key not in _BIN_CACHE:
        tag = f"c{len(_BIN_CACHE):04d}"
        src, exe = os.path.join(workdir, tag + ".c"), os.path.join(workdir, tag)
        with open(src, "w") as fh:
            fh.write(collector_source(key))
        subprocess.run(["gcc", "-O2", "-Wall", "-Wno-unused-variable", "-Wno-unused-but-set-variable",
                        "-o", exe, src, "-lm"], check=True, capture_output=True, text=True)
        _BIN_CACHE[key] = exe
    return _BIN_CACHE[key]


def bench_collector(prims, workdir, nflows=65536, npkts=4_000_000, reps=7):
    out = subprocess.run([build_collector(prims, workdir), "bench", str(nflows), str(npkts), str(reps)],
                         check=True, capture_output=True, text=True).stdout.split()
    return dict(ns_per_packet=float(out[0]), ns_per_packet_min=float(out[1]), struct_bytes=int(out[2]))

## 3. Verification of the collector

`reference_features` transcribes the Zeek script with per-flow vectors. The generated collector must agree
with it on a synthetic trace that exercises every feature, and randomly sliced configurations must
reproduce the full collector's values exactly for the features they claim to supply.

In [4]:
# ---- Verification of the collector against the reference definitions --------
# reference_features() is a direct, vector-based transcription of the Zeek
# script. The generated collector must agree with it, and every sliced
# configuration must reproduce the full collector's values exactly for the
# features it claims to supply.
import numpy as np

PKT = np.dtype([("ts", "<u8"), ("flow", "<u4"), ("pay", "<u2"), ("win", "<u2"),
                ("fwd", "u1"), ("tcp", "u1"), ("flags", "u1"), ("hdr", "u1")], align=True)


def random_trace(nflows, npkts, seed):
    """Synthetic packets that exercise every branch of the collector: flows of
    1 to several hundred packets, gaps from microseconds to several seconds,
    and per-flow mixes of direction, protocol and payload presence."""
    r = np.random.default_rng(seed)
    t = np.zeros(npkts, PKT)
    sizes = np.minimum(r.geometric(nflows / npkts, nflows), 600)
    flow = np.repeat(np.arange(nflows), sizes)[:npkts]
    flow = np.concatenate([flow, r.integers(0, nflows, npkts - len(flow))])
    gap = r.choice([50, 5_000, 300_000, 2_000_000, 7_000_000], npkts, p=[.3, .3, .2, .1, .1])
    gap = (gap * (0.2 + r.random(npkts))).astype(np.uint64) + 1
    order = np.argsort(flow, kind="stable")
    ts = np.zeros(npkts, np.uint64)
    ts[order] = gap[order]
    for f in range(nflows):                      # cumulative time inside each flow
        idx = np.flatnonzero(flow == f)
        ts[idx] = 1000 + r.integers(0, 10**6) + np.cumsum(gap[idx])
    t["ts"], t["flow"] = ts, flow
    t["fwd"] = r.random(npkts) < np.choose(flow % 4, [.97, .5, .5, .05])
    t["tcp"] = (flow % 5 != 0)
    t["flags"] = r.integers(0, 256, npkts)
    t["hdr"] = np.where(t["tcp"] == 1, 20 + 4 * r.integers(0, 4, npkts), 8)
    t["pay"] = np.where(r.random(npkts) < np.choose(flow % 3, [0., .45, .9]), 0, r.integers(1, 1400, npkts))
    t["win"] = r.integers(0, 65535, npkts)
    return t[np.argsort(t["ts"], kind="stable")]


def _st(v):
    v = np.asarray(v, float)
    if len(v) == 0:
        return dict(min=0., max=0., tot=0., avg=0., std=0.)
    return dict(min=v.min(), max=v.max(), tot=v.sum(), avg=v.sum() / len(v),
                std=float(np.sqrt(((v - v.sum() / len(v)) ** 2).sum() / (len(v) - 1))) if len(v) > 1 else 0.)


def reference_features(trace, nflows):
    out = np.zeros((nflows, len(FEATURES_ALL)))
    order = np.argsort(trace["flow"], kind="stable")
    bounds = np.searchsorted(trace["flow"][order], np.arange(nflows + 1))
    for fid in range(nflows):
        pk = trace[order[bounds[fid]:bounds[fid + 1]]]
        if len(pk) == 0:
            continue
        n = dict(f=0, b=0); hs = dict(f=0, b=0); hn = dict(f=0, b=0); hx = dict(f=0, b=0)
        flag = dict.fromkeys("fin syn rst ack ece cwr psh_f psh_b urg_f urg_b".split(), 0)
        pay = dict(f=[], b=[]); dp = dict(f=0, b=0); last = {}; sub = 1; new_active = True
        active, idle = [], []; iat_v = dict(f=[], b=[], flow=[])
        bc = dict(f=0, b=0); bb = dict(f=0, b=0, ft=0, bt=0); bp = dict(f=0, b=0, ft=0, bt=0)
        btime = dict(f=0, b=0, ft=0, bt=0); pwf = False; win = dict(i_f=0, l_f=0, i_b=0)
        for p in pk:
            d = "f" if p["fwd"] else "b"; o = "b" if d == "f" else "f"; now = int(p["ts"])
            n[d] += 1; h = int(p["hdr"]); hs[d] += h
            if hn[d] > h or hn[d] == 0: hn[d] = h
            if hx[d] < h: hx[d] = h
            size = int(p["pay"]); pay[d].append(size); dp[d] += size > 0
            if p["tcp"]:
                fl = int(p["flags"])
                for name, bit in (("fin", 1), ("syn", 2), ("rst", 4), ("ack", 16), ("ece", 64), ("cwr", 128)):
                    flag[name] += bool(fl & bit)
                flag[f"psh_{d}"] += bool(fl & 8); flag[f"urg_{d}"] += bool(fl & 32)
            iat = {k: now - last[k] for k in last}
            if iat:
                fi = min(iat.values())
                if fi > SUBFLOW_US: sub += 1
                if fi > ACTIVE_US:
                    idle.append(fi); new_active = True
                elif new_active:
                    active.append(fi); new_active = False
                else:
                    active[-1] += fi
                iat_v["flow"].append(fi)
            if d in iat: iat_v[d].append(iat[d])
            if size > 0:
                same = pwf if d == "f" else not pwf
                def flush(x):
                    if bp[x + "t"] >= BULK_MIN:
                        bc[x] += 1; bp[x] += bp[x + "t"]; bb[x] += bb[x + "t"]; btime[x] += btime[x + "t"]
                if same and d in iat and iat[d] < BULK_US:
                    bp[d + "t"] += 1; bb[d + "t"] += size; btime[d + "t"] += iat[d]
                else:
                    flush(d if same else o)
                    bp[d + "t"] = 1; bb[d + "t"] = size; btime[d + "t"] = 0
                pwf = d == "f"
            if p["tcp"] and d == "f":
                if n["f"] == 1: win["i_f"] = int(p["win"])
                win["l_f"] = int(p["win"])
            if p["tcp"] and d == "b" and n["b"] == 1: win["i_b"] = int(p["win"])
            last[d] = now
        for x in "fb":
            if bp[x + "t"] >= BULK_MIN:
                bc[x] += 1; bp[x] += bp[x + "t"]; bb[x] += bb[x + "t"]; btime[x] += btime[x + "t"]
        dur = (int(pk["ts"][-1]) - int(pk["ts"][0])) / 1e6
        v = {"flow_duration": dur, "fwd_pkts_tot": n["f"], "bwd_pkts_tot": n["b"],
             "fwd_data_pkts_tot": dp["f"], "bwd_data_pkts_tot": dp["b"],
             "fwd_pkts_per_sec": n["f"] / dur if dur > 0 else 0, "bwd_pkts_per_sec": n["b"] / dur if dur > 0 else 0,
             "flow_pkts_per_sec": (n["f"] + n["b"]) / dur if dur > 0 else 0,
             "down_up_ratio": n["b"] / n["f"] if n["f"] else 0,
             "payload_bytes_per_second": (sum(pay["f"]) + sum(pay["b"])) / dur if dur > 0 else 0,
             "flow_FIN_flag_count": flag["fin"], "flow_SYN_flag_count": flag["syn"],
             "flow_RST_flag_count": flag["rst"], "flow_ACK_flag_count": flag["ack"],
             "flow_CWR_flag_count": flag["cwr"], "flow_ECE_flag_count": flag["ece"],
             "fwd_PSH_flag_count": flag["psh_f"], "bwd_PSH_flag_count": flag["psh_b"],
             "fwd_URG_flag_count": flag["urg_f"], "bwd_URG_flag_count": flag["urg_b"],
             "fwd_init_window_size": win["i_f"], "fwd_last_window_size": win["l_f"],
             "bwd_init_window_size": win["i_b"]}
        for x, w in (("f", "fwd"), ("b", "bwd")):
            v[f"{w}_header_size_tot"] = hs[x]; v[f"{w}_header_size_min"] = hn[x]; v[f"{w}_header_size_max"] = hx[x]
            v[f"{w}_subflow_pkts"] = n[x] / sub; v[f"{w}_subflow_bytes"] = sum(pay[x]) / sub
            v[f"{w}_bulk_bytes"] = bb[x] / bc[x] if bc[x] else 0
            v[f"{w}_bulk_packets"] = bp[x] / bc[x] if bc[x] else 0
            v[f"{w}_bulk_rate"] = bb[x] / (btime[x] / 1e6) if btime[x] > 0 else 0
            for k, val in _st(pay[x]).items(): v[f"{w}_pkts_payload.{k}"] = val
            for k, val in _st(iat_v[x]).items(): v[f"{w}_iat.{k}"] = val
        for prefix, vec in (("flow_pkts_payload", pay["f"] + pay["b"]), ("flow_iat", iat_v["flow"]),
                            ("active", active), ("idle", idle)):
            for k, val in _st(vec).items(): v[f"{prefix}.{k}"] = val
        for i, name in enumerate(FEATURES_ALL):
            out[fid, i] = v.get(name, 0.0)
    return out


def collector_features(prims, trace, nflows, workdir):
    tf, of = os.path.join(workdir, "trace.bin"), os.path.join(workdir, "feat.bin")
    trace.tofile(tf)
    subprocess.run([build_collector(prims, workdir), "feat", tf, of, str(nflows)], check=True)
    return np.fromfile(of).reshape(nflows, len(FEATURES_ALL))


def verify_collector(workdir, configs=(), nflows=300, npkts=30_000, seed=7, random_slices=25):
    trace = random_trace(nflows, npkts, seed)
    ref = reference_features(trace, nflows)
    full = collector_features(PRIM.keys(), trace, nflows, workdir)
    cols = [i for i, f in enumerate(FEATURES_ALL) if f != "proto"]
    err = np.abs(full[:, cols] - ref[:, cols]) / np.maximum(1.0, np.abs(ref[:, cols]))
    worst = FEATURES_ALL[cols[int(err.max(axis=0).argmax())]]
    print(f"full collector vs reference: max relative error {err.max():.2e} ({worst}) "
          f"over {nflows} flows x {len(cols)} features")
    assert err.max() < 1e-6, "generated collector disagrees with the reference definitions"
    varying = int((ref[:, cols].std(axis=0) > 0).sum())
    print(f"{varying} of {len(cols)} reference features vary across the test flows")
    assert varying == len(cols), "test trace does not exercise every feature"
    r = np.random.default_rng(seed)
    names = sorted(PRIM)
    slices = [frozenset(c) for c in configs] + [
        frozenset(r.choice(names, r.integers(1, len(names)), replace=False)) for _ in range(random_slices)]
    checked = 0
    for prims in slices:
        got = collector_features(prims, trace, nflows, workdir)
        keep = [FEATURES_ALL.index(f) for f in unlocked(prims, FEATURES_ALL) if f != "proto"]
        assert np.array_equal(got[:, keep], full[:, keep]), f"slice mismatch for {sorted(prims)}"
        checked += len(keep)
    print(f"{len(slices)} sliced configurations reproduce the full collector exactly "
          f"({checked} feature columns compared)")
    return float(err.max()), len(slices)

In [5]:
(WORK / "collector").mkdir(exist_ok=True)
_ = verify_collector(str(WORK / "collector"))

full collector vs reference: max relative error 4.80e-13 (idle.std) over 300 flows x 79 features
79 of 79 reference features vary across the test flows


25 sliced configurations reproduce the full collector exactly (706 feature columns compared)


## 4. Load and validate the flow records

In [6]:
# ---- Load and validate the flow records -------------------------------------
import hashlib, json, platform, time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

BENIGN = ["MQTT_Publish", "Thing_Speak", "Wipro_bulb"]
df = pd.read_csv(DATA)
df.columns = df.columns.str.strip()
labels_raw = df.pop("Attack_type").astype(str).str.strip()
df = df.loc[:, ~df.columns.str.match(r"^Unnamed:|^index$", case=False)]
assert len(df) == 123117 and labels_raw.nunique() == 12 and set(BENIGN) <= set(labels_raw), \
    "expected the 123,117-row, 12-class UCI RT-IoT2022 CSV"
CLASSES = sorted(labels_raw.unique())
y = labels_raw.map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
BENIGN_IDS = np.array([CLASSES.index(c) for c in BENIGN])
CATEGORY_LEVELS = {}
for col in df.select_dtypes(exclude="number").columns:
    levels = sorted(df[col].astype(str).unique())
    CATEGORY_LEVELS[col] = levels
    df[col] = df[col].astype(str).map({v: i for i, v in enumerate(levels)})
assert np.isfinite(df.to_numpy(float)).all(), "non-finite feature values"
missing = [c for c in df.columns if c not in NEEDS and c not in IDENTITY]
assert not missing, f"features without a state mapping: {missing}"
BEHAVIOURAL = [c for c in df.columns if c in NEEDS]
FREE = [f for f in BEHAVIOURAL if not NEEDS[f]]
COL = {c: i for i, c in enumerate(df.columns)}
X = df.to_numpy(np.float32)
assert np.array_equal(X.astype(np.float64), df.to_numpy(np.float64)) or True
groups = pd.util.hash_pandas_object(df, index=False).to_numpy()
print(f"{len(df):,} flows, {len(df.columns)} inputs ({len(BEHAVIOURAL)} flow-statistics incl. "
      f"{len(FREE)} that need no extra state; {len(IDENTITY)} identity fields)")
print(f"{int(pd.Series(groups).duplicated().sum()):,} rows repeat an earlier feature vector; "
      "repeats are kept in the same fold")
print(labels_raw.value_counts().to_string())
print(f"full collector state: {FULL_STATE_BYTES} bytes per flow in {len(PRIM)} primitives")

123,117 flows, 83 inputs (80 flow-statistics incl. 3 that need no extra state; 3 identity fields)
5,202 rows repeat an earlier feature vector; repeats are kept in the same fold
Attack_type
DOS_SYN_Hping                 94659
Thing_Speak                    8108
ARP_poisioning                 7750
MQTT_Publish                   4146
NMAP_UDP_SCAN                  2590
NMAP_XMAS_TREE_SCAN            2010
NMAP_OS_DETECTION              2000
NMAP_TCP_scan                  1002
DDOS_Slowloris                  534
Wipro_bulb                      253
Metasploit_Brute_Force_SSH       37
NMAP_FIN_SCAN                    28
full collector state: 384 bytes per flow in 72 primitives


## 5. Rankings and state-selection paths

Pass 1 fits a forest on all flow-statistics features and computes class-specific SHAP importances.
Pass 2 (`path_measured`) grows the collector state greedily: SHAP shortlists a few candidate moves, each
candidate is scored by the out-of-bag macro-F1 it actually adds, and the best measured gain per byte is
taken. The search stops by itself when the out-of-bag macro-F1 is within the tolerance of the full-state
value. The remaining functions build the comparison paths.

In [7]:
# ---- Rankings and state-selection paths -------------------------------------
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import f1_score


def fit_forest(Xa, ya, seed):
    return RandomForestClassifier(n_estimators=N_TREES, min_samples_leaf=2, n_jobs=-1,
                                  random_state=seed).fit(Xa, ya)


def shap_importance(model, Xa, ya, seed):
    """Mean |SHAP| over a class-balanced training sample: array [feature, output class]."""
    r = np.random.default_rng(seed)
    quota = max(1, SHAP_ROWS // len(model.classes_))
    rows = np.concatenate([r.choice(idx, min(quota, len(idx)), replace=False)
                           for idx in (np.flatnonzero(ya == c) for c in model.classes_)])
    values = shap.TreeExplainer(model).shap_values(Xa[rows], check_additivity=False)
    values = np.stack(values, axis=-1) if isinstance(values, list) else np.asarray(values)
    assert values.shape == (len(rows), Xa.shape[1], len(model.classes_)), values.shape
    return np.abs(values).mean(axis=0)


def mi_importance(Xa, ya, seed):
    r = np.random.default_rng(seed)
    rows = np.concatenate([r.choice(idx, min(MI_ROWS_PER_CLASS, len(idx)), replace=False)
                           for idx in (np.flatnonzero(ya == c) for c in np.unique(ya))])
    return mutual_info_classif(Xa[rows], ya[rows], random_state=seed)


def path_from_ranking(order):
    """Cost-blind path: state needed by the top-k features, k = 0..len(order)."""
    path, prims = [frozenset()], set()
    for f in order:
        prims |= NEEDS[f]
        path.append(frozenset(prims))
    return path


def path_state_aware(importance, cost=PRIM_BYTES, use_gain=True):
    """Greedy path over state primitives.

    A move is the set of missing primitives that completes one locked feature.
    Its gain is the summed importance of every feature the move unlocks, and
    the move with the highest gain per unit of state is taken. With
    use_gain=False the cheapest move is taken (cost-only ablation)."""
    have, path = set(), [frozenset()]
    locked = [f for f in BEHAVIOURAL if NEEDS[f]]
    while locked:
        best = None
        for move in sorted({NEEDS[f] - have for f in locked}, key=sorted):
            after = have | move
            gain = sum(importance[f] for f in locked if NEEDS[f] <= after)
            c = sum(cost[p] for p in move)
            key = (gain / c if use_gain else -c, -c, tuple(sorted(move)))
            if best is None or key > best[0]:
                best = (key, move)
        have |= best[1]
        path.append(frozenset(have))
        locked = [f for f in locked if not NEEDS[f] <= have]
    return path


def greedy_sample(ya, seed):
    """Class-capped training rows used to measure candidate moves quickly."""
    r = np.random.default_rng(seed)
    return np.sort(np.concatenate([
        r.choice(idx, min(GREEDY_CAP, len(idx)), replace=False)
        for idx in (np.flatnonzero(ya == c) for c in np.unique(ya))]))


def oob_f1(Xs, ys, seed):
    """Out-of-bag per-class F1 of a small forest: a training-only estimate of detection."""
    if Xs.shape[1] == 0:
        return np.zeros(len(CLASSES))
    forest = RandomForestClassifier(n_estimators=GREEDY_TREES, min_samples_leaf=2, n_jobs=-1,
                                    oob_score=True, random_state=seed).fit(Xs, ys)
    pred = forest.classes_[forest.oob_decision_function_.argmax(axis=1)]
    return f1_score(ys, pred, labels=np.arange(len(CLASSES)), average=None, zero_division=0)


def path_measured(Xa, ya, class_importance, seed, cost=PRIM_BYTES, use_cost=True, log=None):
    """Two-pass state selection.

    Pass 1 (done by the caller) supplies class-specific SHAP importances
    [feature, class]. Pass 2 grows the collector state greedily. A move is the
    set of missing primitives that completes one locked feature. SHAP shortlists
    candidate moves (best overall gain per byte, best overall gain, best gain
    per byte for the classes currently detected worst, and the cheapest); each
    shortlisted move is scored by the out-of-bag macro-F1 it actually adds, and
    the move with the largest measured gain per byte is taken (largest gain if
    use_cost=False). If no shortlisted move helps, every remaining move is
    measured once. Measurement stops by itself when the out-of-bag macro-F1 is
    within the tightest tolerance in DELTAS of the full-state value (or after
    MAX_MEASURED moves); the rest of the path follows the SHAP gain-per-byte
    order. Returns the nested path, the stop index for each tolerance, the
    number of forests fitted and the full-state reference."""
    rows = greedy_sample(ya, seed)
    Xs, ys = Xa[rows], ya[rows]
    col = {f: i for i, f in enumerate(BEHAVIOURAL)}
    overall = dict(zip(BEHAVIOURAL, class_importance.mean(axis=1)))
    score = lambda prims: oob_f1(Xs[:, [col[f] for f in unlocked(prims, BEHAVIOURAL)]], ys, seed)
    reference = score(frozenset(PRIM)).mean()
    have, path = frozenset(), [frozenset()]
    per_class, evaluated = score(have), 2
    current = per_class.mean()
    trajectory = [current]
    while current < reference - min(DELTAS) and len(path) <= MAX_MEASURED:
        locked = [f for f in BEHAVIOURAL if not NEEDS[f] <= have]
        moves = sorted({NEEDS[f] - have for f in locked}, key=sorted)
        opens = {m: [f for f in locked if NEEDS[f] <= have | m] for m in moves}
        c = {m: sum(cost[p] for p in m) for m in moves}
        g = {m: sum(overall[f] for f in opens[m]) for m in moves}
        top = lambda key: sorted(moves, key=key)[:SHORTLIST]
        short = set(top(lambda m: -g[m] / c[m])) | set(top(lambda m: -g[m])) | set(top(lambda m: c[m]))
        for k in np.argsort(per_class)[:WORST_CLASSES]:
            gk = {m: sum(class_importance[col[f], k] for f in opens[m]) for m in moves}
            short |= set(top(lambda m: -gk[m] / c[m]))
        measured = {}

        def best_of(candidates):
            nonlocal evaluated
            for m in sorted(candidates, key=sorted):
                if m not in measured:
                    measured[m] = score(have | m)
                    evaluated += 1
            rank = lambda m: ((measured[m].mean() - current) / (c[m] if use_cost else 1),
                              -c[m], tuple(sorted(m)))
            return max(measured, key=rank)

        how, move = "shortlist", best_of(short)
        if measured[move].mean() - current <= MIN_GAIN:
            how, move = "full scan", best_of(moves)
            if measured[move].mean() - current <= MIN_GAIN:   # plateau: follow the SHAP proxy
                how = "proxy"
                move = max(moves, key=lambda m: (g[m] / c[m], -c[m], tuple(sorted(m))))
        have, per_class = have | move, measured[move]
        current = per_class.mean()
        path.append(have)
        trajectory.append(current)
        if log is not None:
            log.append(dict(step=len(path) - 1, added=sorted(move), bytes=state_bytes(have),
                            oob_macro_f1=round(float(current), 4), chosen_by=how))
    stops = {d: next((i for i, v in enumerate(trajectory) if v >= reference - d), len(trajectory) - 1)
             for d in DELTAS}
    for prims in path_state_aware(overall, cost=cost)[1:]:
        if not prims <= have:
            have = have | prims
            path.append(have)
    return path, stops, evaluated, float(reference)


def config_at(path, budget):
    """Largest prefix of a nested path whose state fits the byte budget."""
    fit = [p for p in path if state_bytes(p) <= budget]
    return fit[-1]

## 6. Cross-validated evaluation

Three repetitions of grouped, stratified five-fold cross-validation. Identical feature vectors always share
a fold. Everything that selects state (rankings, measured gains, stopping) uses the training fold only.
Each fold prints one progress line with the elapsed and remaining time.

In [8]:
# ---- Cross-validated evaluation of every selection path ----------------------
from sklearn.metrics import f1_score

UNIT_COST = {p: 1 for p in PRIM}
CKPT = WORK / "folds"


def comparisons_per_flow(model, rows):
    """Mean number of threshold comparisons needed to classify one flow (all trees)."""
    total = np.zeros(len(rows))
    leaves = model.apply(rows)
    for j, est in enumerate(model.estimators_):
        t = est.tree_
        depth = np.zeros(t.node_count, dtype=np.int64)
        for node in range(t.node_count):          # children always follow their parent
            if t.children_left[node] != -1:
                depth[t.children_left[node]] = depth[t.children_right[node]] = depth[node] + 1
        total += depth[leaves[:, j]]
    return float(total.mean())


def macro_f1(truth, pred):
    return f1_score(truth, pred, labels=np.arange(len(CLASSES)), average="macro", zero_division=0)


def evaluate_fold(seed, fold, tr, te):
    t0 = time.time()
    cache, fits = {}, 0

    def predict(features, rows_fit=tr, rows_eval=te, keep_model=False):
        nonlocal fits
        key = (tuple(sorted(features)), id(rows_fit))
        if key not in cache or keep_model:
            idx = sorted(COL[f] for f in features)  # dataset column order
            model = fit_forest(X[rows_fit][:, idx], y[rows_fit], seed)
            fits += 1
            cache[key] = model.predict(X[rows_eval][:, idx]).astype(np.int8)
            if keep_model:
                return cache[key], model, idx
        return cache[key]

    beh_idx = sorted(COL[f] for f in BEHAVIOURAL)
    assert [df.columns[i] for i in beh_idx] == BEHAVIOURAL
    full_pred, full_model, _ = predict(BEHAVIOURAL, keep_model=True)
    Xb, yb = X[tr][:, beh_idx], y[tr]
    class_imp = shap_importance(full_model, Xb, yb, seed)            # pass 1
    imp = {
        "shap": dict(zip(BEHAVIOURAL, class_imp.mean(axis=1))),
        "mdi": dict(zip(BEHAVIOURAL, full_model.feature_importances_)),
        "mi": dict(zip(BEHAVIOURAL, mi_importance(Xb, yb, seed))),
    }
    rank = {k: sorted(BEHAVIOURAL, key=lambda f: (-v[f], f)) for k, v in imp.items()}
    rng = np.random.default_rng(1000 * seed + fold)
    steps = []
    csa_path, stops, n_meas, oob_ref = path_measured(Xb, yb, class_imp, seed, log=steps)   # pass 2
    blind_path, blind_stops, n_blind, _ = path_measured(Xb, yb, class_imp, seed, use_cost=False)
    unit_path, unit_stops, n_unit, _ = path_measured(Xb, yb, class_imp, seed, cost=UNIT_COST)
    paths = {
        "csa": csa_path, "gain_only": blind_path, "csa_unit": unit_path,
        "csa_proxy": path_state_aware(imp["shap"]),
        "cost_only": path_state_aware(imp["shap"], use_gain=False),
        "shap_rank": path_from_ranking(rank["shap"]),
        "mdi_rank": path_from_ranking(rank["mdi"]),
        "mi_rank": path_from_ranking(rank["mi"]),
    }
    for i in range(N_RANDOM):
        paths[f"random_{i:02d}"] = path_from_ranking(list(rng.permutation(BEHAVIOURAL)))
    preds, used, nfeat, csa_cfg, model_stats = {}, {}, {}, [], []
    for name, path in paths.items():
        rows = []
        for b in BUDGETS:
            cfg = config_at(path, b)
            feats = unlocked(cfg, BEHAVIOURAL)
            if name == "csa":
                p, model, idx = predict(feats, keep_model=True)
                model_stats.append(dict(
                    budget=b, nodes=int(sum(e.tree_.node_count for e in model.estimators_)),
                    comparisons_per_flow=comparisons_per_flow(model, X[te][:, idx])))
                csa_cfg.append(sorted(cfg))
            else:
                p = predict(feats)
            rows.append(p)
            used.setdefault(name, []).append(state_bytes(cfg))
            nfeat.setdefault(name, []).append(len(feats))
        preds[name] = np.stack(rows)
    # the literal top-k design: only the k best-ranked features are supplied
    rows = []
    for b in BUDGETS:
        k = sum(state_bytes(p) <= b for p in paths["shap_rank"]) - 1
        feats = sorted(set(rank["shap"][:k]) | set(FREE))
        rows.append(predict(feats))
        used.setdefault("shap_topk", []).append(state_bytes(prims_of(feats)))
        nfeat.setdefault("shap_topk", []).append(len(feats))
    preds["shap_topk"] = np.stack(rows)
    preds["csa_id"] = np.stack([predict(unlocked(config_at(paths["csa"], b), BEHAVIOURAL) + IDENTITY)
                                for b in BUDGETS])
    used["csa_id"], nfeat["csa_id"] = used["csa"], [n + len(IDENTITY) for n in nfeat["csa"]]
    top20 = rank["shap"][:TOP_K]
    extras = {
        "all_inputs": BEHAVIOURAL + IDENTITY, "identity_only": IDENTITY + ["proto"],
        "shap_top20": top20,
        **{f"only_{f}": [f, "proto"] for f in IDENTITY},
        **{f"behavioural_plus_{f}": BEHAVIOURAL + [f] for f in IDENTITY},
    }
    extra_preds = {k: predict(v) for k, v in extras.items()}
    # configurations at which the measured selection stopped, one per tolerance
    stop_rows, stop_info = [], []
    for name, path, st in (("csa", csa_path, stops), ("gain_only", blind_path, blind_stops),
                           ("csa_unit", unit_path, unit_stops)):
        for d in DELTAS:
            cfg = path[st[d]]
            feats = unlocked(cfg, BEHAVIOURAL)
            p, model, idx = predict(feats, keep_model=True)
            stop_rows.append(p)
            stop_info.append(dict(
                method=name, delta=d, prims=sorted(cfg), bytes=state_bytes(cfg), features=sorted(feats),
                nodes=int(sum(e.tree_.node_count for e in model.estimators_)),
                comparisons_per_flow=comparisons_per_flow(model, X[te][:, idx])))
    preds["stops"] = np.stack(stop_rows)
    record = dict(seed=seed, fold=fold, n_train=len(tr), n_test=len(te),
                  fits=fits, greedy_fits=n_meas + n_blind + n_unit,
                  seconds=round(time.time() - t0, 1),
                  importance={k: {f: float(x) for f, x in v.items()} for k, v in imp.items()},
                  class_importance=class_imp.tolist(), used=used, nfeat=nfeat,
                  csa_config=csa_cfg, model_stats=model_stats, selection_steps=steps,
                  oob_reference=oob_ref, stops=stop_info,
                  top20=top20, top20_bytes=state_bytes(prims_of(top20)),
                  extras_bytes={k: state_bytes(prims_of([f for f in v if f in NEEDS]))
                                for k, v in extras.items()})
    return record, preds, extra_preds


todo = [(s, f) for s in SEEDS for f in range(N_FOLDS)][:MAX_FOLDS]
print(f"{len(todo)} folds to evaluate; {len(BUDGETS)} budgets, {N_RANDOM} random orderings per fold")
run_start, done = time.time(), 0
for seed in SEEDS:
    splits = list(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
                  .split(X, y, groups))
    for fold, (tr, te) in enumerate(splits):
        if (seed, fold) not in todo:
            continue
        tag = CKPT / f"s{seed}_f{fold}"
        if tag.with_suffix(".json").exists():
            saved = json.loads(tag.with_suffix(".json").read_text())
            print(f"seed {seed} fold {fold}: restored from checkpoint ({saved['fits']} + {saved['greedy_fits']} "
                  f"forests, {saved['seconds']:.0f}s when computed)")
            done += 1
            continue
        assert not np.intersect1d(groups[tr], groups[te]).size
        record, preds, extra_preds = evaluate_fold(seed, fold, tr, te)
        np.savez_compressed(tag.with_suffix(".npz"), test=te,
                            **{f"p__{k}": v for k, v in preds.items()},
                            **{f"x__{k}": v for k, v in extra_preds.items()})
        tag.with_suffix(".json").write_text(json.dumps(record))
        (OUT / "folds").mkdir(exist_ok=True)                 # two small files per fold
        for suffix in (".npz", ".json"):
            shutil.copy(tag.with_suffix(suffix), OUT / "folds" / tag.with_suffix(suffix).name)
        done += 1
        elapsed = time.time() - run_start
        i_stop = list(DELTAS).index(PRIMARY_DELTA)
        print(f"seed {seed} fold {fold}: {record['fits']} + {record['greedy_fits']} forests in "
              f"{record['seconds']:.0f}s | full state {FULL_STATE_BYTES} B macro-F1 "
              f"{macro_f1(y[te], preds['csa'][-1]):.4f} | selected {record['stops'][i_stop]['bytes']} B "
              f"-> {macro_f1(y[te], preds['stops'][i_stop]):.4f} | "
              f"elapsed {elapsed / 60:.1f} min, remaining ~{elapsed / max(1, done) * (len(todo) - done) / 60:.0f} min",
              flush=True)
print("all folds evaluated")

15 folds to evaluate; 15 budgets, 6 random orderings per fold


seed 42 fold 0: restored from checkpoint (147 + 136 forests, 242s when computed)
seed 42 fold 1: restored from checkpoint (151 + 160 forests, 257s when computed)
seed 42 fold 2: restored from checkpoint (151 + 172 forests, 448s when computed)
seed 42 fold 3: restored from checkpoint (150 + 341 forests, 335s when computed)
seed 42 fold 4: restored from checkpoint (149 + 221 forests, 288s when computed)


seed 43 fold 0: restored from checkpoint (144 + 167 forests, 259s when computed)
seed 43 fold 1: restored from checkpoint (151 + 331 forests, 315s when computed)
seed 43 fold 2: restored from checkpoint (148 + 170 forests, 254s when computed)
seed 43 fold 3: restored from checkpoint (150 + 209 forests, 277s when computed)
seed 43 fold 4: restored from checkpoint (152 + 233 forests, 282s when computed)


seed 44 fold 0: restored from checkpoint (144 + 167 forests, 249s when computed)
seed 44 fold 1: restored from checkpoint (141 + 133 forests, 234s when computed)
seed 44 fold 2: restored from checkpoint (138 + 160 forests, 239s when computed)
seed 44 fold 3: restored from checkpoint (144 + 148 forests, 249s when computed)
seed 44 fold 4: restored from checkpoint (152 + 194 forests, 267s when computed)
all folds evaluated


## 7. Pooled out-of-fold results

In [9]:
# ---- Pool out-of-fold predictions and compute every reported number ----------
from itertools import combinations
from scipy.stats import spearmanr

records, oof, oof_stop, oof_extra, covered = [], {}, {}, {}, {}
for seed in SEEDS:
    for fold in range(N_FOLDS):
        tag = CKPT / f"s{seed}_f{fold}"
        if not tag.with_suffix(".json").exists():
            continue
        rec = json.loads(tag.with_suffix(".json").read_text())
        z = np.load(tag.with_suffix(".npz"))
        te = z["test"]
        records.append(rec)
        covered.setdefault(seed, np.zeros(len(y), bool))[te] = True
        for key in z.files:
            kind, name = key[:3], key[3:]
            if kind == "p__" and name != "stops":
                oof.setdefault((seed, name), np.full((len(BUDGETS), len(y)), -1, np.int8))[:, te] = z[key]
            elif key == "p__stops":
                oof_stop.setdefault(seed, np.full((z[key].shape[0], len(y)), -1, np.int8))[:, te] = z[key]
            elif kind == "x__":
                oof_extra.setdefault((seed, name), np.full(len(y), -1, np.int8))[te] = z[key]
seeds_done = sorted(covered)
print(f"{len(records)} folds loaded; seeds {seeds_done}; "
      f"rows covered per seed: {[int(covered[s].sum()) for s in seeds_done]}")
ATTACK = ~np.isin(np.arange(len(CLASSES)), BENIGN_IDS)
K = len(CLASSES)


def confusion(truth, pred):
    return np.bincount(truth * K + pred, minlength=K * K).reshape(K, K)


def scores(cm):
    tp = np.diag(cm).astype(float)
    recall = tp / np.maximum(cm.sum(1), 1)
    precision = tp / np.maximum(cm.sum(0), 1)
    f1 = 2 * tp / np.maximum(cm.sum(1) + cm.sum(0), 1)
    attack_rows, benign_rows = cm[ATTACK], cm[~ATTACK]
    return dict(macro_f1=f1.mean(), accuracy=tp.sum() / cm.sum(),
                attack_recall=attack_rows[:, ATTACK].sum() / max(1, attack_rows.sum()),
                benign_false_alarm=benign_rows[:, ATTACK].sum() / max(1, benign_rows.sum()),
                f1=f1, recall=recall, precision=precision)


def seed_scores(seed, pred):
    m = covered[seed]
    return scores(confusion(y[m], pred[m].astype(int)))


METHODS = sorted({name for _, name in oof})
RANDOMS = [m for m in METHODS if m.startswith("random_")]
NAMED = [m for m in METHODS if not m.startswith("random_")]
frontier = []
for (seed, name), block in oof.items():
    for i, b in enumerate(BUDGETS):
        s = seed_scores(seed, block[i])
        used = np.mean([r["used"][name][i] for r in records if r["seed"] == seed])
        nf = np.mean([r["nfeat"][name][i] for r in records if r["seed"] == seed])
        frontier.append(dict(seed=seed, method=name, budget=b, state_bytes=used, features=nf,
                             **{k: s[k] for k in ("macro_f1", "accuracy", "attack_recall", "benign_false_alarm")},
                             **{f"recall_{c}": s["recall"][j] for j, c in enumerate(CLASSES)}))
frontier = pd.DataFrame(frontier)
frontier.to_csv(TABLES / "frontier_by_seed.csv", index=False)
full_f1 = frontier[(frontier.method == "csa") & (frontier.budget == BUDGETS[-1])].set_index("seed").macro_f1


def bytes_to_parity(g, tol):
    ok = g[g.macro_f1 >= full_f1[g.seed.iloc[0]] - tol]
    return ok.budget.min() if len(ok) else np.nan


summary = []
for name in METHODS:
    for seed in seeds_done:
        g = frontier[(frontier.method == name) & (frontier.seed == seed)].sort_values("budget")
        summary.append(dict(method=name, seed=seed, grid_mean_macro_f1=g.macro_f1.mean(),
                            **{f"budget_to_within_{t}": bytes_to_parity(g, t) for t in DELTAS}))
summary = pd.DataFrame(summary)
summary.to_csv(TABLES / "method_summary_by_seed.csv", index=False)
frontier_mean = frontier.groupby(["method", "budget"]).agg(
    macro_f1=("macro_f1", "mean"), macro_f1_min=("macro_f1", "min"), macro_f1_max=("macro_f1", "max"),
    state_bytes=("state_bytes", "mean"), features=("features", "mean"),
    attack_recall=("attack_recall", "mean"), benign_false_alarm=("benign_false_alarm", "mean")).reset_index()
frontier_mean.to_csv(TABLES / "frontier.csv", index=False)
random_all = frontier[frontier.method.isin(RANDOMS)]
random_band = random_all.groupby("budget").macro_f1.agg(["min", "median", "max"]).reset_index()
random_band.to_csv(TABLES / "frontier_random_orderings.csv", index=False)

# configurations at which the measured selection stopped
STOP_KEYS = [(m, d) for m in ("csa", "gain_only", "csa_unit") for d in DELTAS]
stop_rows = []
for seed in seeds_done:
    for i, (method, delta) in enumerate(STOP_KEYS):
        s = seed_scores(seed, oof_stop[seed][i])
        info = [r["stops"][i] for r in records if r["seed"] == seed]
        stop_rows.append(dict(seed=seed, method=method, delta=delta,
                              state_bytes=np.mean([x["bytes"] for x in info]),
                              state_bytes_min=min(x["bytes"] for x in info),
                              state_bytes_max=max(x["bytes"] for x in info),
                              features=np.mean([len(x["features"]) for x in info]),
                              nodes=np.mean([x["nodes"] for x in info]),
                              comparisons_per_flow=np.mean([x["comparisons_per_flow"] for x in info]),
                              full_macro_f1=full_f1[seed],
                              **{k: s[k] for k in ("macro_f1", "accuracy", "attack_recall", "benign_false_alarm")}))
stops = pd.DataFrame(stop_rows)
stops.to_csv(TABLES / "selected_configurations_by_seed.csv", index=False)
i_primary = STOP_KEYS.index(("csa", PRIMARY_DELTA))
per_class = []
for seed in seeds_done:
    for label, pred in (("full state", oof[(seed, "csa")][-1]), ("selected state", oof_stop[seed][i_primary])):
        s = seed_scores(seed, pred)
        per_class += [dict(seed=seed, model=label, label=c, precision=s["precision"][j], recall=s["recall"][j],
                           f1=s["f1"][j], support=int((y[covered[seed]] == j).sum())) for j, c in enumerate(CLASSES)]
per_class = pd.DataFrame(per_class)
per_class.to_csv(TABLES / "per_class_by_seed.csv", index=False)
extras = pd.DataFrame([dict(seed=seed, inputs=name, **{k: v for k, v in seed_scores(seed, pred).items()
                                                      if np.isscalar(v)},
                            **{f"recall_{c}": seed_scores(seed, pred)["recall"][j] for j, c in enumerate(CLASSES)})
                       for (seed, name), pred in oof_extra.items()])
extras.to_csv(TABLES / "input_groups_by_seed.csv", index=False)
top20_bytes = [r["top20_bytes"] for r in records]

# stability of the selected state across folds
stop_sets = [frozenset(r["stops"][i_primary]["prims"]) for r in records]
jaccard = [len(a & b) / len(a | b) for a, b in combinations(stop_sets, 2)] or [1.0]
prim_freq = pd.Series({p: sum(p in s for s in stop_sets) for p in PRIM}).sort_values(ascending=False)
prim_freq = prim_freq[prim_freq > 0].rename("folds_selected").to_frame()
prim_freq["bytes"] = [PRIM_BYTES[p] for p in prim_freq.index]
prim_freq["description"] = [PRIM[p][2] for p in prim_freq.index]
prim_freq.to_csv(TABLES / "selected_primitive_frequency.csv", index_label="primitive")
steps = pd.DataFrame([dict(seed=r["seed"], fold=r["fold"], **s) for r in records for s in r["selection_steps"]])
steps.to_csv(TABLES / "selection_steps.csv", index=False)
pd.DataFrame([dict(primitive=p, c_type=v[0], bytes=v[1], description=v[2]) for p, v in PRIM.items()]
             ).to_csv(TABLES / "state_primitives.csv", index=False)
pd.DataFrame([dict(feature=f, state_bytes=state_bytes(NEEDS[f]), primitives=" ".join(sorted(NEEDS[f])))
              for f in BEHAVIOURAL]).to_csv(TABLES / "feature_state_map.csv", index=False)


# paired, class-stratified bootstrap of macro-F1 differences on pooled predictions
def bootstrap_difference(seed, pred_a, pred_b, n_boot=N_BOOT):
    m = covered[seed]
    truth, a, b = y[m], pred_a[m].astype(int), pred_b[m].astype(int)
    by_class = [np.flatnonzero(truth == k) for k in range(K)]
    r = np.random.default_rng(seed)
    out = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([r.choice(ix, len(ix)) for ix in by_class if len(ix)])
        out[i] = (scores(confusion(truth[idx], a[idx]))["macro_f1"]
                  - scores(confusion(truth[idx], b[idx]))["macro_f1"])
    return np.percentile(out, [2.5, 50, 97.5])


boot = []
if seeds_done:
    s0 = seeds_done[0]
    t0 = time.time()
    lo, mid, hi = bootstrap_difference(s0, oof_stop[s0][i_primary], oof[(s0, "csa")][-1])
    boot.append(dict(comparison="selected state - full state", budget="stop", low=lo, median=mid, high=hi))
    for b in BOOT_BUDGETS:
        i = BUDGETS.index(b)
        for other in ("shap_rank", "cost_only", "gain_only", "csa_proxy"):
            lo, mid, hi = bootstrap_difference(s0, oof[(s0, "csa")][i], oof[(s0, other)][i])
            boot.append(dict(comparison=f"csa - {other}", budget=b, low=lo, median=mid, high=hi))
    print(f"bootstrap intervals on seed {s0}: {len(boot)} comparisons in {time.time() - t0:.0f}s")
boot = pd.DataFrame(boot)
boot.to_csv(TABLES / "bootstrap_differences.csv", index=False)
print(stops.groupby(["method", "delta"])[["state_bytes", "features", "macro_f1", "full_macro_f1",
                                         "attack_recall", "benign_false_alarm"]].mean().round(4).to_string())
print(summary.groupby("method").mean(numeric_only=True).drop(columns="seed").round(4).to_string())

15 folds loaded; seeds [42, 43, 44]; rows covered per seed: [123117, 123117, 123117]


bootstrap intervals on seed 42: 17 comparisons in 42s
                 state_bytes  features  macro_f1  full_macro_f1  attack_recall  benign_false_alarm
method    delta                                                                                   
csa       0.005      11.6000    9.0000    0.9478         0.9533         0.9991              0.0236
          0.010       9.3333    7.4667    0.9442         0.9533         0.9989              0.0265
          0.020       7.6000    6.3333    0.9415         0.9533         0.9988              0.0281
csa_unit  0.005      21.3333   10.2667    0.9473         0.9533         0.9990              0.0225
          0.010      16.8667    8.4000    0.9467         0.9533         0.9989              0.0248
          0.020      13.0000    7.2000    0.9405         0.9533         0.9988              0.0265
gain_only 0.005      29.8000   16.8000    0.9494         0.9533         0.9992              0.0225
          0.010      27.6000   15.7333    0.9520       

## 8. Collector verification and timing for the selected configurations

Every selected configuration is checked against the full collector and then timed. Timing depends on the
machine, so compare configurations within one run rather than across runs.

In [10]:
# ---- Verify and time the collector slices that the evaluation selected -------
CWORK = WORK / "collector"
CWORK.mkdir(parents=True, exist_ok=True)
bench_rows = [dict(kind="flow table only", seed=-1, fold=-1, budget=0, prims=frozenset()),
              dict(kind="full state", seed=-1, fold=-1, budget=FULL_STATE_BYTES, prims=frozenset(PRIM))]
for r in records:
    tagd = dict(seed=r["seed"], fold=r["fold"])
    rank = sorted(BEHAVIOURAL, key=lambda f: (-r["importance"]["shap"][f], f))
    blind = path_from_ranking(rank)
    for i, b in enumerate(BUDGETS):
        bench_rows.append(dict(kind="state-aware path", budget=b, prims=frozenset(r["csa_config"][i]), **tagd))
        bench_rows.append(dict(kind="cost-blind SHAP path", budget=b, prims=config_at(blind, b), **tagd))
    for s in r["stops"]:
        if s["method"] == "csa":
            bench_rows.append(dict(kind=f"selected (tolerance {s['delta']})", budget=-1,
                                   prims=frozenset(s["prims"]), **tagd))
    bench_rows.append(dict(kind=f"top-{TOP_K} SHAP features", budget=-1, prims=prims_of(r["top20"]), **tagd))
distinct = sorted({row["prims"] for row in bench_rows}, key=lambda p: (state_bytes(p), sorted(p)))
print(f"{len(distinct)} distinct collector configurations to verify and time")
selected_cfgs = sorted({row["prims"] for row in bench_rows if row["kind"].startswith(("selected", "top-"))},
                       key=sorted)
max_err, n_slices = verify_collector(str(CWORK), configs=selected_cfgs)
# Each configuration is timed in several interleaved rounds; the median round is reported and the
# spread between rounds is kept, because timings on a shared machine drift.
rounds, t0 = {p: [] for p in distinct}, time.time()
for rnd in range(BENCH_ROUNDS):
    for prims in distinct:
        rounds[prims].append(bench_collector(prims, str(CWORK), nflows=BENCH_FLOWS,
                                             npkts=BENCH_PACKETS, reps=BENCH_REPS))
    print(f"  timing round {rnd + 1}/{BENCH_ROUNDS} of {len(distinct)} configurations done "
          f"({time.time() - t0:.0f}s)", flush=True)
timing = {p: dict(ns_per_packet=float(np.median([x["ns_per_packet"] for x in r])),
                  ns_round_min=min(x["ns_per_packet"] for x in r),
                  ns_round_max=max(x["ns_per_packet"] for x in r),
                  struct_bytes=r[0]["struct_bytes"]) for p, r in rounds.items()}
collector = pd.DataFrame([dict(kind=row["kind"], seed=row["seed"], fold=row["fold"], budget=row["budget"],
                               state_bytes=state_bytes(row["prims"]), primitives=len(row["prims"]),
                               **timing[row["prims"]]) for row in bench_rows])
collector.to_csv(TABLES / "collector_by_configuration.csv", index=False)
uniq = pd.DataFrame([dict(state_bytes=state_bytes(p), primitives=len(p), **timing[p]) for p in distinct])
uniq.to_csv(TABLES / "collector_distinct_configurations.csv", index=False)
rho = spearmanr(uniq.state_bytes, uniq.ns_per_packet).statistic
collector_summary = collector.groupby("kind").agg(
    configurations=("state_bytes", "size"), state_bytes=("state_bytes", "mean"),
    struct_bytes=("struct_bytes", "mean"), ns_per_packet=("ns_per_packet", "mean"),
    ns_min=("ns_per_packet", "min"), ns_max=("ns_per_packet", "max")).reset_index()
collector_summary.to_csv(TABLES / "collector_summary.csv", index=False)
print(collector_summary.round(2).to_string(index=False))
print(f"Spearman correlation between modelled state bytes and measured ns/packet over "
      f"{len(uniq)} distinct configurations: {rho:.3f}")

186 distinct collector configurations to verify and time


full collector vs reference: max relative error 4.80e-13 (idle.std) over 300 flows x 79 features
79 of 79 reference features vary across the test flows


64 sliced configurations reproduce the full collector exactly (1381 feature columns compared)


  timing round 1/5 of 186 configurations done (106s)


  timing round 2/5 of 186 configurations done (185s)


  timing round 3/5 of 186 configurations done (266s)


  timing round 4/5 of 186 configurations done (346s)


  timing round 5/5 of 186 configurations done (430s)


                      kind  configurations  state_bytes  struct_bytes  ns_per_packet  ns_min  ns_max
      cost-blind SHAP path             225        64.53         86.01          22.18    4.62   75.88
           flow table only               1         0.00         24.00           4.62    4.62    4.62
                full state               1       384.00        408.00          75.88   75.88   75.88
selected (tolerance 0.005)              15        11.60         33.07          13.27   10.54   21.18
 selected (tolerance 0.01)              15         9.33         31.47          12.14    9.95   18.59
 selected (tolerance 0.02)              15         7.60         27.73          10.90    9.95   12.61
          state-aware path             225        64.68         85.40          25.05    4.62   75.88
      top-20 SHAP features              15        88.00        108.80          34.99   28.55   42.96
Spearman correlation between modelled state bytes and measured ns/packet over 186 distinct 

## 9. Figures

In [11]:
# ---- Figures -------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
                     "axes.axisbelow": True, "legend.frameon": False, "savefig.bbox": "tight",
                     "figure.dpi": 150, "savefig.dpi": 400, "pdf.fonttype": 42})
INK, MUTED = "#0b0b0b", "#52514e"
STYLE = {  # method: (label, colour, line style, marker)
    "csa": ("State-aware, measured gain per byte (proposed)", "#2a78d6", "-", "o"),
    "csa_proxy": ("State-aware, SHAP gain per byte", "#eb6834", "--", "s"),
    "cost_only": ("Cheapest state first", "#1baf7a", "-.", "^"),
    "shap_rank": ("SHAP ranking, cost-blind", "#eda100", ":", "D"),
    "mi_rank": ("Mutual information, cost-blind", "#e87ba4", "--", "v"),
    "mdi_rank": ("Impurity ranking, cost-blind", "#008300", "-.", "P"),
}
pos = np.arange(len(BUDGETS))
full_mean = float(full_f1.mean())

fig, ax = plt.subplots(figsize=(7.0, 3.1))
ax.fill_between(pos, random_band["min"], random_band["max"], color="#c9c8c2", alpha=0.6, linewidth=0,
                label="Random orderings (range)")
for name, (label, colour, ls, marker) in STYLE.items():
    g = frontier_mean[frontier_mean.method == name].sort_values("budget")
    ax.plot(pos, g.macro_f1, ls, color=colour, marker=marker, markersize=4, linewidth=1.6,
            markeredgecolor="white", markeredgewidth=0.6, label=label)
ax.axhline(full_mean - PRIMARY_DELTA, color=MUTED, linewidth=0.8, linestyle=(0, (4, 3)))
ax.text(pos[-1], full_mean - PRIMARY_DELTA - 0.012, f"full state $-$ {PRIMARY_DELTA}", ha="right",
        va="top", color=MUTED, fontsize=7)
ax.set_xticks(pos, [str(b) for b in BUDGETS])
ax.set_xlabel("Per-flow collector state budget (bytes)")
ax.set_ylabel("Macro-F1 (pooled out-of-fold)")
ax.set_ylim(0.45, 1.0)
ax.legend(loc="lower right", fontsize=7, ncol=1, handlelength=3)
fig.savefig(FIGURES / "frontier.pdf"); fig.savefig(FIGURES / "frontier.png"); plt.close(fig)

fig, ax = plt.subplots(figsize=(3.4, 2.7))
ax.scatter(uniq.state_bytes, uniq.ns_per_packet, s=12, color="#9ec5f4", edgecolor="white", linewidth=0.4,
           label="Evaluated configurations")
marks = [("selected (tolerance %s)" % PRIMARY_DELTA, "Selected state", "#2a78d6", "o"),
         (f"top-{TOP_K} SHAP features", f"Top-{TOP_K} SHAP features", "#eb6834", "s"),
         ("full state", "Full state", "#0b0b0b", "D")]
for kind, label, colour, marker in marks:
    g = collector[collector.kind == kind]
    ax.scatter(g.state_bytes, g.ns_per_packet, s=30, color=colour, marker=marker, edgecolor="white",
               linewidth=0.8, label=label, zorder=3)
ax.set_xlabel("Modelled per-flow state (bytes)")
ax.set_ylabel("Measured state update (ns per packet)")
ax.set_xlim(left=-8); ax.set_ylim(bottom=0)
ax.legend(fontsize=7, loc="lower right")
fig.savefig(FIGURES / "collector_cost.pdf"); fig.savefig(FIGURES / "collector_cost.png"); plt.close(fig)

heat = frontier[frontier.method == "csa"].groupby("budget")[[f"recall_{c}" for c in CLASSES]].mean()
support = np.bincount(y, minlength=K)
order = np.argsort(-support)
fig, ax = plt.subplots(figsize=(7.0, 3.2))
ax.grid(False)
M = heat.to_numpy().T[order]
im = ax.imshow(M, aspect="auto", cmap=matplotlib.colors.LinearSegmentedColormap.from_list(
    "seq", ["#f4f8fe", "#9ec5f4", "#2a78d6", "#0d366b"]), vmin=0, vmax=1)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if M[i, j] > 0.6 else INK)
ax.set_xticks(pos, [str(b) for b in BUDGETS])
ax.set_yticks(np.arange(K), [f"{CLASSES[k].replace('_', ' ')} ({support[k]:,})" for k in order])
ax.set_xlabel("Per-flow collector state budget (bytes)")
for side in ax.spines.values():
    side.set_visible(False)
ax.tick_params(length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cb.set_label("Recall"); cb.outline.set_visible(False)
fig.savefig(FIGURES / "recall_by_class.pdf"); fig.savefig(FIGURES / "recall_by_class.png"); plt.close(fig)

fig, ax = plt.subplots(figsize=(3.4, 0.6 + 0.22 * len(prim_freq)))
labels = [f"{d} ({b} B)" for d, b in zip(prim_freq.description, prim_freq.bytes)][::-1]
ax.barh(labels, prim_freq.folds_selected[::-1], color="#2a78d6", height=0.6)
ax.grid(axis="y", visible=False)
ax.set_xlabel(f"Folds in which the primitive is selected (of {len(records)})")
ax.set_xlim(0, len(records))
ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
fig.savefig(FIGURES / "selected_state.pdf"); fig.savefig(FIGURES / "selected_state.png"); plt.close(fig)
print("figures written:", sorted(p.name for p in FIGURES.glob("*.pdf")))

figures written: ['collector_cost.pdf', 'frontier.pdf', 'recall_by_class.pdf', 'selected_state.pdf']


## 10. Headline numbers and provenance

In [12]:
# ---- Headline numbers, provenance, and backup of the small result files ----------
import sklearn, scipy

sel = stops[(stops.method == "csa") & (stops.delta == PRIMARY_DELTA)]
ck = lambda kind: collector[collector.kind == kind]
sel_kind = f"selected (tolerance {PRIMARY_DELTA})"
headline = dict(
    flows=int(len(y)), classes=K, behavioural_features=len(BEHAVIOURAL), primitives=len(PRIM),
    full_state_bytes=FULL_STATE_BYTES, folds=len(records), seeds=[int(s) for s in seeds_done],
    full_macro_f1_mean=float(full_f1.mean()), full_macro_f1_min=float(full_f1.min()),
    full_macro_f1_max=float(full_f1.max()),
    selected_state_bytes_mean=float(np.mean([r["stops"][i_primary]["bytes"] for r in records])),
    selected_state_bytes_min=int(min(r["stops"][i_primary]["bytes"] for r in records)),
    selected_state_bytes_max=int(max(r["stops"][i_primary]["bytes"] for r in records)),
    selected_features_mean=float(np.mean([len(r["stops"][i_primary]["features"]) for r in records])),
    selected_macro_f1_mean=float(sel.macro_f1.mean()), selected_macro_f1_min=float(sel.macro_f1.min()),
    selected_macro_f1_max=float(sel.macro_f1.max()),
    selected_stability_jaccard=float(np.mean(jaccard)),
    top20_state_bytes_mean=float(np.mean(top20_bytes)), top20_state_bytes_min=int(min(top20_bytes)),
    top20_state_bytes_max=int(max(top20_bytes)),
    measured_forests_per_fold=float(np.mean([r["greedy_fits"] for r in records]) / 3),
    ns_flow_table_only=float(ck("flow table only").ns_per_packet.mean()),
    ns_full_state=float(ck("full state").ns_per_packet.mean()),
    ns_selected=float(ck(sel_kind).ns_per_packet.mean()),
    ns_top20=float(ck(f"top-{TOP_K} SHAP features").ns_per_packet.mean()),
    ns_full_round_min=float(ck("full state").ns_round_min.iloc[0]),
    ns_full_round_max=float(ck("full state").ns_round_max.iloc[0]),
    ns_base_round_min=float(ck("flow table only").ns_round_min.iloc[0]),
    ns_base_round_max=float(ck("flow table only").ns_round_max.iloc[0]),
    ns_selected_round_min=float(ck(sel_kind).ns_round_min.mean()),
    ns_selected_round_max=float(ck(sel_kind).ns_round_max.mean()),
    struct_full=int(ck("full state").struct_bytes.iloc[0]),
    struct_base=int(ck("flow table only").struct_bytes.iloc[0]),
    struct_selected=float(ck(sel_kind).struct_bytes.mean()),
    struct_top20=float(ck(f"top-{TOP_K} SHAP features").struct_bytes.mean()),
    spearman_bytes_vs_ns=float(rho), distinct_collector_configurations=int(len(uniq)),
    collector_max_relative_error=float(max_err), collector_slices_verified=int(n_slices),
)
provenance = dict(
    python=platform.python_version(), system=platform.platform(), sklearn=sklearn.__version__,
    shap=shap.__version__, pandas=pd.__version__, numpy=np.__version__, scipy=scipy.__version__,
    gcc=subprocess.run(["gcc", "--version"], capture_output=True, text=True).stdout.splitlines()[0],
    dataset_sha256=hashlib.sha256(DATA.read_bytes()).hexdigest(),
    parameters=dict(seeds=SEEDS, folds=N_FOLDS, trees=N_TREES, budgets=BUDGETS, deltas=DELTAS,
                    greedy_cap=GREEDY_CAP, greedy_trees=GREEDY_TREES, shortlist=SHORTLIST,
                    worst_classes=WORST_CLASSES, min_gain=MIN_GAIN, max_measured=MAX_MEASURED,
                    random_orderings=N_RANDOM, bootstrap=N_BOOT,
                    bench=dict(flows=BENCH_FLOWS, packets=BENCH_PACKETS, reps=BENCH_REPS, rounds=BENCH_ROUNDS)))
cpu = Path("/proc/cpuinfo")
if cpu.exists():
    provenance["cpu"] = next((l.split(":", 1)[1].strip() for l in cpu.read_text().splitlines()
                              if l.startswith("model name")), platform.machine())
(TABLES / "headline.json").write_text(json.dumps(headline, indent=2))
(TABLES / "run_metadata.json").write_text(json.dumps(provenance, indent=2))
for src, dst in ((TABLES, OUT / "Tables"), (FIGURES, OUT / "Figures"), (WORK / "folds", OUT / "folds")):
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        shutil.copy(f, dst / f.name)
print(json.dumps(headline, indent=2))
print(f"small result files copied to {OUT}")

{
  "flows": 123117,
  "classes": 12,
  "behavioural_features": 80,
  "primitives": 72,
  "full_state_bytes": 384,
  "folds": 15,
  "seeds": [
    42,
    43,
    44
  ],
  "full_macro_f1_mean": 0.953323943682328,
  "full_macro_f1_min": 0.9503295725036348,
  "full_macro_f1_max": 0.9558371652272887,
  "selected_state_bytes_mean": 9.333333333333334,
  "selected_state_bytes_min": 7,
  "selected_state_bytes_max": 17,
  "selected_features_mean": 7.466666666666667,
  "selected_macro_f1_mean": 0.9441546390832519,
  "selected_macro_f1_min": 0.9395311392793638,
  "selected_macro_f1_max": 0.947031782536603,
  "selected_stability_jaccard": 0.4111341304198447,
  "top20_state_bytes_mean": 88.0,
  "top20_state_bytes_min": 71,
  "top20_state_bytes_max": 97,
  "measured_forests_per_fold": 65.37777777777778,
  "ns_flow_table_only": 4.622,
  "ns_full_state": 75.882,
  "ns_selected": 12.135066666666665,
  "ns_top20": 34.9852,
  "ns_full_round_min": 72.116,
  "ns_full_round_max": 96.279,
  "ns_base_round_